# 05. Event Detection & Classification

**Objective:** Detect and classify 12 types of financial events from news articles

**Event Types:**
1. Earnings Announcements
2. Mergers & Acquisitions
3. Product Launches
4. Regulatory Actions
5. Executive Changes
6. Legal Issues
7. Partnership Deals
8. Market Disruptions
9. Dividend Announcements
10. Restructuring
11. Analyst Ratings
12. Economic Indicators

**Methods:**
- Rule-based classification with keyword matching
- Regex pattern matching for structured events
- Multi-label classification (articles can have multiple events)
- Event impact scoring

In [1]:
import pandas as pd
import numpy as np
import json
import re
from pathlib import Path
from collections import Counter, defaultdict
from tqdm.auto import tqdm
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# NLP
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import MultiLabelBinarizer

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(" All libraries loaded successfully")

 All libraries loaded successfully


## 1. Setup Paths and Configuration

In [2]:
# Setup paths
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / '01_DATA'
FEATURES_DIR = DATA_DIR / 'features'
RESULTS_DIR = BASE_DIR / '03_RESULTS'
OUTPUTS_DIR = RESULTS_DIR / 'outputs'
VIZ_DIR = RESULTS_DIR / 'visualizations'
CONFIG_DIR = BASE_DIR / '06_CONFIG'

# Create directories
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# Load configurations
with open(CONFIG_DIR / 'framework_config.json', 'r') as f:
    config = json.load(f)

with open(CONFIG_DIR / 'financial_events.json', 'r') as f:
    events_config = json.load(f)

with open(CONFIG_DIR / 'event_id_mapping.json', 'r') as f:
    event_mapping = json.load(f)

with open(CONFIG_DIR / 'financial_lexicon.json', 'r') as f:
    lexicon = json.load(f)

print(f" Base directory: {BASE_DIR}")
print(f" Features directory: {FEATURES_DIR}")
print(f" Results directory: {RESULTS_DIR}")
print(f" Configurations loaded")
print(f"\n Event types configured: {len(events_config['events'])}")

 Base directory: c:\Users\HARSHIT\Desktop\p\nlp analysis
 Features directory: c:\Users\HARSHIT\Desktop\p\nlp analysis\01_DATA\features
 Results directory: c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS
 Configurations loaded

 Event types configured: 12


## 2. Load Entity-Enriched Dataset

In [3]:
# Load dataset with entities from notebook 04
df = pd.read_csv(FEATURES_DIR / 'dataset_with_entities.csv')
df['date'] = pd.to_datetime(df['date'])

print(f" Loaded {len(df):,} articles")
print(f" Date range: {df['date'].min()} to {df['date'].max()}")
print(f" Companies: {df['ticker'].nunique()}")

df.head()

 Loaded 63 articles
 Date range: 2025-09-26 07:00:00 to 2026-01-18 12:19:48
 Companies: 19


,date,title,text,source,url,ticker,publisher,query,article_length,word_count,...,textblob_polarity,textblob_label,date_only,spacy_entities_json,bert_entities_json,all_entities_json,num_entities,num_orgs,num_persons,num_locations
0,2026-01-18 12:19:48,American Express Company $AXP Shares Acquired ...,American Express Company $AXP Shares Acquired ...,Google News,https://news.google.com/rss/articles/CBMi2wFBV...,AXP,NaN,AXP stock news,106,13,...,0.000000,neutral,2026-01-18,"[{""text"": ""American Express Company $AXP Share...","[{""text"": ""American Express Company"", ""label"":...","[{""text"": ""American Express Company $AXP Share...",7,6,0,1
1,2026-01-18 12:15:12,Apple stock price slips to $255 ahead of a hol...,Apple stock price slips to $255 ahead of a hol...,Google News,https://news.google.com/rss/articles/CBMiuAFBV...,AAPL,NaN,AAPL stock news,121,18,...,0.000000,neutral,2026-01-18,"[{""text"": ""Apple"", ""label"": ""ORG"", ""start"": 0,...","[{""text"": ""Apple"", ""label"": ""ORG"", ""score"": 0....","[{""text"": ""Apple"", ""label"": ""ORG"", ""source"": ""...",5,4,0,0
2,2026-01-18 12:00:26,Amgen Inc. (NASDAQ:AMGN) is largely controlled...,Amgen Inc. (NASDAQ:AMGN) is largely controlled...,Google News,https://news.google.com/rss/articles/CBMigAFBV...,AMGN,NaN,AMGN stock news,128,16,...,0.407143,positive,2026-01-18,"[{""text"": ""Amgen Inc."", ""label"": ""ORG"", ""start...","[{""text"": ""Amgen Inc"", ""label"": ""ORG"", ""score""...","[{""text"": ""Amgen Inc."", ""label"": ""ORG"", ""sourc...",7,5,1,0
3,2026-01-18 11:43:58,BAC Stock Gains On Bullish Q4 Print Guides For...,BAC Stock Gains On Bullish Q4 Print Guides For...,Google News,https://news.google.com/rss/articles/CBMisgFBV...,BAC,NaN,BAC stock news,100,15,...,0.000000,neutral,2026-01-18,"[{""text"": ""BAC Stock Gains On Bullish Q4 Print...",[],"[{""text"": ""BAC Stock Gains On Bullish Q4 Print...",2,1,0,1
4,2026-01-18 11:26:14,Strong Analyst Confidence in Alibaba Group Hol...,Strong Analyst Confidence in Alibaba Group Hol...,Google News,https://news.google.com/rss/articles/CBMiswFBV...,BABA,NaN,BABA stock news,102,13,...,0.433333,positive,2026-01-18,"[{""text"": ""Alibaba Group Holding"", ""label"": ""O...","[{""text"": ""Alibaba Group Holding"", ""label"": ""O...","[{""text"": ""Alibaba Group Holding"", ""label"": ""O...",5,5,0,0


## 3. Build Event Detection Rules

In [4]:
class EventDetector:
    """Rule-based event detection with keyword and pattern matching"""
    
    def __init__(self, events_config, event_mapping):
        self.events_config = events_config
        self.event_mapping = event_mapping
        self.event_types = event_mapping['event_types']
        
        # Build compiled regex patterns for each event type
        self.patterns = {}
        for event in events_config['events']:
            event_name = event['name']
            patterns = []
            
            # Compile regex patterns
            for pattern in event.get('patterns', []):
                try:
                    patterns.append(re.compile(pattern, re.IGNORECASE))
                except:
                    pass
            
            self.patterns[event_name] = {
                'patterns': patterns,
                'keywords': [kw.lower() for kw in event.get('keywords', [])],
                'id': event['id']
            }
    
    def detect_events(self, text, title=''):
        """Detect events in text using rules and patterns"""
        if pd.isna(text):
            return []
        
        text_lower = str(text).lower()
        title_lower = str(title).lower() if title else ''
        combined_text = title_lower + ' ' + text_lower
        
        detected_events = []
        
        for event_name, rules in self.patterns.items():
            score = 0
            
            # Check patterns (higher weight)
            for pattern in rules['patterns']:
                if pattern.search(combined_text):
                    score += 3
            
            # Check keywords (lower weight)
            for keyword in rules['keywords']:
                if keyword in combined_text:
                    score += 1
            
            # If score is high enough, add event
            if score >= 2:  # Threshold: need pattern match or multiple keywords
                detected_events.append({
                    'event_name': event_name,
                    'event_id': rules['id'],
                    'confidence': min(score / 5, 1.0)  # Normalize to 0-1
                })
        
        return detected_events

# Initialize detector
detector = EventDetector(events_config, event_mapping)

# Test on sample texts
test_texts = [
    "Apple reports Q1 earnings beat with record iPhone sales exceeding analyst estimates",
    "Microsoft acquires Activision Blizzard for $69 billion in gaming industry megadeal",
    "Tesla launches new Cybertruck after years of production delays",
    "FDA approves Pfizer's new drug for treatment of rare disease"
]

print(" Testing event detection:")
print("=" * 80)
for text in test_texts:
    events = detector.detect_events(text)
    print(f"\nText: {text[:80]}...")
    print(f"Events: {[e['event_name'] for e in events]}")

print("\n Event detector initialized")

 Testing event detection:

Text: Apple reports Q1 earnings beat with record iPhone sales exceeding analyst estima...
Events: []

Text: Microsoft acquires Activision Blizzard for $69 billion in gaming industry megade...
Events: ['MERGER_ACQUISITION']

Text: Tesla launches new Cybertruck after years of production delays...
Events: ['PRODUCT_LAUNCH']

Text: FDA approves Pfizer's new drug for treatment of rare disease...
Events: ['REGULATORY_ACTION']

 Event detector initialized


## 4. Detect Events in All Articles

In [5]:
print(" Detecting events in all articles...")

# Detect events for each article
events_list = []
for idx, row in tqdm(df.iterrows(), total=len(df), desc="Detecting events"):
    events = detector.detect_events(
        text=row.get('text', ''),
        title=row.get('title', '')
    )
    events_list.append(events)

df['detected_events'] = events_list

# Extract event information
df['num_events'] = df['detected_events'].apply(len)
df['event_names'] = df['detected_events'].apply(lambda x: [e['event_name'] for e in x])
df['event_ids'] = df['detected_events'].apply(lambda x: [e['event_id'] for e in x])
df['primary_event'] = df['detected_events'].apply(
    lambda x: x[0]['event_name'] if len(x) > 0 else 'NO_EVENT'
)
df['primary_event_id'] = df['detected_events'].apply(
    lambda x: x[0]['event_id'] if len(x) > 0 else -1
)

# Statistics
articles_with_events = (df['num_events'] > 0).sum()
total_events = df['num_events'].sum()

print(f"\n Event detection complete!")
print(f"\n Detection statistics:")
print(f"  Articles with events: {articles_with_events:,} ({articles_with_events/len(df)*100:.1f}%)")
print(f"  Total events detected: {total_events:,}")
print(f"  Average events per article: {df['num_events'].mean():.2f}")
print(f"  Articles with multiple events: {(df['num_events'] > 1).sum():,}")

 Detecting events in all articles...


Detecting events:   0%|          | 0/63 [00:00<?, ?it/s]


 Event detection complete!

 Detection statistics:
  Articles with events: 1 (1.6%)
  Total events detected: 1
  Average events per article: 0.02
  Articles with multiple events: 0


## 5. Event Distribution Analysis

In [6]:
# Count event types
all_event_names = []
for events in df['event_names']:
    all_event_names.extend(events)

event_counts = Counter(all_event_names)

print(" Event Type Distribution:")
print("=" * 60)
for event_name, count in event_counts.most_common():
    percentage = count / len(df) * 100
    print(f"{event_name:30s}: {count:6,} ({percentage:5.2f}%)")

# Create DataFrame for visualization
event_dist_df = pd.DataFrame([
    {'event': k, 'count': v, 'percentage': v/len(df)*100}
    for k, v in event_counts.most_common()
])

# Save event distribution
event_dist_df.to_csv(OUTPUTS_DIR / 'event_distribution.csv', index=False)
print(f"\n Saved event distribution to {OUTPUTS_DIR / 'event_distribution.csv'}")

 Event Type Distribution:
EARNINGS_ANNOUNCEMENT         :      1 ( 1.59%)

 Saved event distribution to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\event_distribution.csv


## 6. Visualize Event Distribution

In [7]:
# Bar chart of event distribution
fig = go.Figure(data=[
    go.Bar(
        y=event_dist_df['event'][::-1],
        x=event_dist_df['count'][::-1],
        orientation='h',
        marker=dict(
            color=event_dist_df['count'][::-1],
            colorscale='Viridis',
            showscale=True,
            colorbar=dict(title="Count")
        ),
        text=event_dist_df['count'][::-1],
        textposition='auto',
    )
])

fig.update_layout(
    title="Financial Event Types Detected in News Articles",
    xaxis_title="Number of Articles",
    yaxis_title="Event Type",
    height=600,
    showlegend=False
)

fig.write_html(VIZ_DIR / 'event_distribution.html')
fig.show()

# Pie chart
fig2 = px.pie(
    event_dist_df,
    values='count',
    names='event',
    title='Event Type Distribution (Proportion)',
    hole=0.3
)

fig2.update_traces(textposition='inside', textinfo='percent+label')
fig2.write_html(VIZ_DIR / 'event_distribution_pie.html')
fig2.show()

print(" Visualizations saved")

 Visualizations saved


## 7. Temporal Event Analysis

In [8]:
# Events over time
df['year_month'] = df['date'].dt.to_period('M').astype(str)

# Create event time series
event_timeline = []
for idx, row in df.iterrows():
    for event_name in row['event_names']:
        event_timeline.append({
            'date': row['date'],
            'year_month': row['year_month'],
            'event': event_name,
            'ticker': row.get('ticker', 'UNKNOWN')
        })

event_timeline_df = pd.DataFrame(event_timeline)

if len(event_timeline_df) > 0:
    # Monthly event counts by type
    event_monthly = event_timeline_df.groupby(['year_month', 'event']).size().reset_index(name='count')
    
    # Plot top 5 event types over time
    top_events = event_counts.most_common(5)
    top_event_names = [e[0] for e in top_events]
    
    event_monthly_filtered = event_monthly[event_monthly['event'].isin(top_event_names)]
    
    fig = px.line(
        event_monthly_filtered,
        x='year_month',
        y='count',
        color='event',
        title='Top 5 Event Types Over Time',
        labels={'year_month': 'Month', 'count': 'Number of Events'}
    )
    
    fig.update_layout(
        height=500,
        xaxis_tickangle=-45,
        hovermode='x unified'
    )
    
    fig.write_html(VIZ_DIR / 'event_timeline.html')
    fig.show()
    
    print(" Temporal analysis complete")
else:
    print(" No events to analyze temporally")

 Temporal analysis complete


## 8. Event-Sentiment Correlation

In [9]:
# Analyze sentiment by event type
event_sentiment = []
for idx, row in df.iterrows():
    for event_name in row['event_names']:
        event_sentiment.append({
            'event': event_name,
            'finbert_sentiment': row.get('finbert_sentiment', 0),
            'vader_compound': row.get('vader_compound', 0),
            'textblob_polarity': row.get('textblob_polarity', 0)
        })

event_sentiment_df = pd.DataFrame(event_sentiment)

if len(event_sentiment_df) > 0:
    # Average sentiment by event type
    event_sent_avg = event_sentiment_df.groupby('event').agg({
        'finbert_sentiment': 'mean',
        'vader_compound': 'mean',
        'textblob_polarity': 'mean'
    }).reset_index()
    
    event_sent_avg = event_sent_avg.sort_values('finbert_sentiment', ascending=False)
    
    print(" Average Sentiment by Event Type:")
    print("=" * 80)
    print(f"{'Event Type':30s} {'FinBERT':>10s} {'VADER':>10s} {'TextBlob':>10s}")
    print("=" * 80)
    for idx, row in event_sent_avg.iterrows():
        print(f"{row['event']:30s} {row['finbert_sentiment']:>10.3f} {row['vader_compound']:>10.3f} {row['textblob_polarity']:>10.3f}")
    
    # Visualize
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        y=event_sent_avg['event'],
        x=event_sent_avg['finbert_sentiment'],
        name='FinBERT',
        orientation='h'
    ))
    
    fig.add_trace(go.Bar(
        y=event_sent_avg['event'],
        x=event_sent_avg['vader_compound'],
        name='VADER',
        orientation='h'
    ))
    
    fig.update_layout(
        title="Average Sentiment by Event Type",
        xaxis_title="Sentiment Score",
        yaxis_title="Event Type",
        height=600,
        barmode='group'
    )
    
    fig.write_html(VIZ_DIR / 'event_sentiment_correlation.html')
    fig.show()
    
    # Save
    event_sent_avg.to_csv(OUTPUTS_DIR / 'event_sentiment_averages.csv', index=False)
    print(f"\n Saved to {OUTPUTS_DIR / 'event_sentiment_averages.csv'}")
else:
    print(" No event-sentiment data to analyze")

 Average Sentiment by Event Type:
Event Type                        FinBERT      VADER   TextBlob
EARNINGS_ANNOUNCEMENT               0.273      0.273      0.000



 Saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\event_sentiment_averages.csv


## 9. Event Co-occurrence Analysis

In [10]:
# Find which events occur together
from itertools import combinations

event_pairs = Counter()
for events in df['event_names']:
    if len(events) > 1:
        for pair in combinations(sorted(events), 2):
            event_pairs[pair] += 1

if len(event_pairs) > 0:
    print(" Top Event Co-occurrences:")
    print("=" * 80)
    for (event1, event2), count in event_pairs.most_common(20):
        print(f"{event1} + {event2}: {count:,} articles")
    
    # Create co-occurrence matrix
    event_types_list = list(set([e for events in df['event_names'] for e in events]))
    n_events = len(event_types_list)
    
    cooccurrence_matrix = np.zeros((n_events, n_events))
    event_to_idx = {event: idx for idx, event in enumerate(event_types_list)}
    
    for events in df['event_names']:
        for i, event1 in enumerate(events):
            for event2 in events[i+1:]:
                idx1 = event_to_idx[event1]
                idx2 = event_to_idx[event2]
                cooccurrence_matrix[idx1, idx2] += 1
                cooccurrence_matrix[idx2, idx1] += 1
    
    # Visualize heatmap
    fig = px.imshow(
        cooccurrence_matrix,
        x=event_types_list,
        y=event_types_list,
        color_continuous_scale='YlOrRd',
        title='Event Co-occurrence Matrix',
        labels=dict(color="Co-occurrences")
    )
    
    fig.update_layout(
        height=800,
        xaxis_tickangle=-45
    )
    
    fig.write_html(VIZ_DIR / 'event_cooccurrence_matrix.html')
    fig.show()
    
    # Save co-occurrences
    cooccur_df = pd.DataFrame([
        {'event1': pair[0], 'event2': pair[1], 'count': count}
        for pair, count in event_pairs.most_common()
    ])
    cooccur_df.to_csv(OUTPUTS_DIR / 'event_cooccurrences.csv', index=False)
    print(f"\n Saved to {OUTPUTS_DIR / 'event_cooccurrences.csv'}")
else:
    print(" No event co-occurrences found")

 No event co-occurrences found


## 10. Event Impact Scoring

In [11]:
# Calculate event impact scores based on:
# 1. Event type importance (from config)
# 2. Sentiment magnitude
# 3. Number of entities mentioned
# 4. Article prominence (title vs body)

def calculate_event_impact(row):
    """Calculate impact score for article based on events and features"""
    if row['num_events'] == 0:
        return 0.0
    
    # Get event impact levels
    event_impact_map = events_config.get('event_impact', {})
    impact_scores = {'HIGH': 1.0, 'MEDIUM': 0.6, 'LOW': 0.3}
    
    # Base impact from event types
    base_impact = 0
    for event_name in row['event_names']:
        impact_level = event_impact_map.get(event_name, 'MEDIUM')
        base_impact += impact_scores.get(impact_level, 0.5)
    base_impact = base_impact / row['num_events']  # Average
    
    # Sentiment magnitude (absolute value)
    sentiment_magnitude = abs(row.get('finbert_sentiment', 0))
    
    # Entity richness (normalized)
    entity_score = min(row.get('num_entities', 0) / 20, 1.0)
    
    # Combined impact score
    impact = (
        base_impact * 0.5 +           # Event type importance: 50%
        sentiment_magnitude * 0.3 +    # Sentiment magnitude: 30%
        entity_score * 0.2             # Entity richness: 20%
    )
    
    return min(impact, 1.0)  # Cap at 1.0

df['event_impact_score'] = df.apply(calculate_event_impact, axis=1)

# Categorize impact
df['impact_category'] = pd.cut(
    df['event_impact_score'],
    bins=[-0.01, 0.3, 0.6, 1.0],
    labels=['LOW', 'MEDIUM', 'HIGH']
)

print(" Event Impact Distribution:")
print(df['impact_category'].value_counts())
print(f"\nAverage impact score: {df['event_impact_score'].mean():.3f}")
print(f"Median impact score: {df['event_impact_score'].median():.3f}")

# Top impact articles
print("\n Top 10 Highest Impact Events:")
top_impact = df.nlargest(10, 'event_impact_score')[['date', 'title', 'primary_event', 'event_impact_score']]
for idx, row in top_impact.iterrows():
    print(f"\n{row['date'].date()} | Impact: {row['event_impact_score']:.3f}")
    print(f"  Event: {row['primary_event']}")
    print(f"  Title: {row['title'][:100]}...")

 Event Impact Distribution:
impact_category
LOW       62
MEDIUM     1
HIGH       0
Name: count, dtype: int64

Average impact score: 0.007
Median impact score: 0.000

 Top 10 Highest Impact Events:

2025-11-08 | Impact: 0.452
  Event: EARNINGS_ANNOUNCEMENT
  Title: Why APA (APA) Is Up 5.2% After Returning to Profitability and Raising Production Guidance - Yahoo Fi...

2026-01-18 | Impact: 0.000
  Event: NO_EVENT
  Title: American Express Company $AXP Shares Acquired by Nations Financial Group Inc. IA ADV - MarketBeat...

2026-01-18 | Impact: 0.000
  Event: NO_EVENT
  Title: Apple stock price slips to $255 ahead of a holiday-shortened week — AAPL earnings are the next test ...

2026-01-18 | Impact: 0.000
  Event: NO_EVENT
  Title: Amgen Inc. (NASDAQ:AMGN) is largely controlled by institutional shareholders who own 80% of the comp...

2026-01-18 | Impact: 0.000
  Event: NO_EVENT
  Title: BAC Stock Gains On Bullish Q4 Print Guides For Mid-Single Digit NII Growth In FY26 - Menafn...

2026-0

## 11. Company Event Profile

In [12]:
# Analyze events by company
company_events = []
for ticker in df['ticker'].unique():
    if pd.isna(ticker):
        continue
    
    ticker_df = df[df['ticker'] == ticker]
    
    # Count events
    ticker_events = []
    for events in ticker_df['event_names']:
        ticker_events.extend(events)
    
    event_counter = Counter(ticker_events)
    
    company_events.append({
        'ticker': ticker,
        'total_articles': len(ticker_df),
        'total_events': len(ticker_events),
        'avg_impact': ticker_df['event_impact_score'].mean(),
        'most_common_event': event_counter.most_common(1)[0][0] if event_counter else 'NONE',
        'event_diversity': len(event_counter),  # Number of different event types
        'high_impact_events': (ticker_df['impact_category'] == 'HIGH').sum()
    })

company_events_df = pd.DataFrame(company_events).sort_values('total_events', ascending=False)

print(" Top 20 Companies by Event Activity:")
print("=" * 100)
print(f"{'Ticker':8s} {'Articles':>10s} {'Events':>10s} {'Avg Impact':>12s} {'Event Diversity':>15s} {'Most Common Event':>25s}")
print("=" * 100)
for idx, row in company_events_df.head(20).iterrows():
    print(f"{row['ticker']:8s} {row['total_articles']:10,} {row['total_events']:10,} "
          f"{row['avg_impact']:12.3f} {row['event_diversity']:15} {row['most_common_event']:>25s}")

# Save
company_events_df.to_csv(OUTPUTS_DIR / 'company_event_profiles.csv', index=False)
print(f"\n Saved to {OUTPUTS_DIR / 'company_event_profiles.csv'}")

 Top 20 Companies by Event Activity:
Ticker     Articles     Events   Avg Impact Event Diversity         Most Common Event
APA               4          1        0.113               1     EARNINGS_ANNOUNCEMENT
AXP               2          0        0.000               0                      NONE
BBD               2          0        0.000               0                      NONE
AEP               1          0        0.000               0                      NONE
AVB               3          0        0.000               0                      NONE
ADBE              2          0        0.000               0                      NONE
AMZN              2          0        0.000               0                      NONE
BA                4          0        0.000               0                      NONE
ASML              2          0        0.000               0                      NONE
AVGO              6          0        0.000               0                      NONE
AAPL             

## 12. Save Enhanced Dataset

In [13]:
# Convert event lists to JSON strings for CSV storage
df['detected_events_json'] = df['detected_events'].apply(json.dumps)
df['event_names_json'] = df['event_names'].apply(json.dumps)
df['event_ids_json'] = df['event_ids'].apply(json.dumps)

# Create binary columns for each event type (for ML later)
for event_info in events_config['events']:
    event_name = event_info['name']
    df[f'has_{event_name}'] = df['event_names'].apply(lambda x: 1 if event_name in x else 0)

# Save to CSV (drop original list columns)
df_to_save = df.drop(columns=['detected_events', 'event_names', 'event_ids'])
df_to_save.to_csv(FEATURES_DIR / 'dataset_with_events.csv', index=False)

print(f"\n Saved enhanced dataset to {FEATURES_DIR / 'dataset_with_events.csv'}")
print(f" Dataset shape: {df_to_save.shape}")
print(f"\n New event columns added:")
print(f"  - detected_events_json, event_names_json, event_ids_json")
print(f"  - num_events, primary_event, primary_event_id")
print(f"  - event_impact_score, impact_category")
print(f"  - Binary indicators for each event type (has_*)")

df_to_save.head()


 Saved enhanced dataset to c:\Users\HARSHIT\Desktop\p\nlp analysis\01_DATA\features\dataset_with_events.csv
 Dataset shape: (63, 47)

 New event columns added:
  - detected_events_json, event_names_json, event_ids_json
  - num_events, primary_event, primary_event_id
  - event_impact_score, impact_category
  - Binary indicators for each event type (has_*)


,date,title,text,source,url,ticker,publisher,query,article_length,word_count,...,has_PRODUCT_LAUNCH,has_REGULATORY_ACTION,has_EXECUTIVE_CHANGE,has_LEGAL_ISSUE,has_PARTNERSHIP_DEAL,has_MARKET_DISRUPTION,has_DIVIDEND_ANNOUNCEMENT,has_RESTRUCTURING,has_ANALYST_RATING,has_ECONOMIC_INDICATOR
0,2026-01-18 12:19:48,American Express Company $AXP Shares Acquired ...,American Express Company $AXP Shares Acquired ...,Google News,https://news.google.com/rss/articles/CBMi2wFBV...,AXP,NaN,AXP stock news,106,13,...,0,0,0,0,0,0,0,0,0,0
1,2026-01-18 12:15:12,Apple stock price slips to $255 ahead of a hol...,Apple stock price slips to $255 ahead of a hol...,Google News,https://news.google.com/rss/articles/CBMiuAFBV...,AAPL,NaN,AAPL stock news,121,18,...,0,0,0,0,0,0,0,0,0,0
2,2026-01-18 12:00:26,Amgen Inc. (NASDAQ:AMGN) is largely controlled...,Amgen Inc. (NASDAQ:AMGN) is largely controlled...,Google News,https://news.google.com/rss/articles/CBMigAFBV...,AMGN,NaN,AMGN stock news,128,16,...,0,0,0,0,0,0,0,0,0,0
3,2026-01-18 11:43:58,BAC Stock Gains On Bullish Q4 Print Guides For...,BAC Stock Gains On Bullish Q4 Print Guides For...,Google News,https://news.google.com/rss/articles/CBMisgFBV...,BAC,NaN,BAC stock news,100,15,...,0,0,0,0,0,0,0,0,0,0
4,2026-01-18 11:26:14,Strong Analyst Confidence in Alibaba Group Hol...,Strong Analyst Confidence in Alibaba Group Hol...,Google News,https://news.google.com/rss/articles/CBMiswFBV...,BABA,NaN,BABA stock news,102,13,...,0,0,0,0,0,0,0,0,0,0


## 13. Summary Statistics

In [14]:
# Create comprehensive summary
summary = {
    'total_articles': len(df),
    'articles_with_events': int((df['num_events'] > 0).sum()),
    'coverage_rate': float((df['num_events'] > 0).sum() / len(df)),
    'total_events_detected': int(df['num_events'].sum()),
    'avg_events_per_article': float(df['num_events'].mean()),
    'event_distribution': dict(event_counts),
    'impact_distribution': {
        'HIGH': int((df['impact_category'] == 'HIGH').sum()),
        'MEDIUM': int((df['impact_category'] == 'MEDIUM').sum()),
        'LOW': int((df['impact_category'] == 'LOW').sum())
    },
    'avg_impact_score': float(df['event_impact_score'].mean()),
    'companies_analyzed': int(df['ticker'].nunique()),
    'top_10_events': event_counts.most_common(10),
    'event_sentiment_correlation': event_sent_avg.to_dict('records') if len(event_sentiment_df) > 0 else [],
    'date_range': {
        'start': str(df['date'].min()),
        'end': str(df['date'].max())
    }
}

# Save summary
with open(OUTPUTS_DIR / 'event_detection_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(" Event Detection & Classification Summary")
print("=" * 70)
print(f"Total articles processed: {summary['total_articles']:,}")
print(f"Articles with events: {summary['articles_with_events']:,} ({summary['coverage_rate']*100:.1f}%)")
print(f"Total events detected: {summary['total_events_detected']:,}")
print(f"Average events per article: {summary['avg_events_per_article']:.2f}")
print(f"\nImpact distribution:")
print(f"  HIGH: {summary['impact_distribution']['HIGH']:,}")
print(f"  MEDIUM: {summary['impact_distribution']['MEDIUM']:,}")
print(f"  LOW: {summary['impact_distribution']['LOW']:,}")
print(f"\nAverage impact score: {summary['avg_impact_score']:.3f}")

print(f"\n Summary saved to {OUTPUTS_DIR / 'event_detection_summary.json'}")
print("\n Event Detection & Classification complete!")
print("\n Output files:")
print(f"  - {FEATURES_DIR / 'dataset_with_events.csv'}")
print(f"  - {OUTPUTS_DIR / 'event_distribution.csv'}")
print(f"  - {OUTPUTS_DIR / 'event_sentiment_averages.csv'}")
print(f"  - {OUTPUTS_DIR / 'event_cooccurrences.csv'}")
print(f"  - {OUTPUTS_DIR / 'company_event_profiles.csv'}")
print(f"  - {VIZ_DIR / 'event_distribution.html'}")
print(f"  - {VIZ_DIR / 'event_timeline.html'}")
print(f"  - {VIZ_DIR / 'event_sentiment_correlation.html'}")
print(f"  - {VIZ_DIR / 'event_cooccurrence_matrix.html'}")

 Event Detection & Classification Summary
Total articles processed: 63
Articles with events: 1 (1.6%)
Total events detected: 1
Average events per article: 0.02

Impact distribution:
  HIGH: 0
  MEDIUM: 1
  LOW: 62

Average impact score: 0.007

 Summary saved to c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\event_detection_summary.json

 Event Detection & Classification complete!

 Output files:
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\01_DATA\features\dataset_with_events.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\event_distribution.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\event_sentiment_averages.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\event_cooccurrences.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\outputs\company_event_profiles.csv
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\visualizations\event_distribution.html
  - c:\Users\HARSHIT\Desktop\p\nlp analysis\03_RESULTS\